# NORIA — Stage 2 proof (photoreal talking Noria)

This runs **SadTalker** (open-source, free) on Google's free GPU to turn Noria's
real render + a voice line into a **photoreal talking video**. No cost, no account
beyond a Google login.

**How to run:** `Runtime → Run all`. First set the GPU: `Runtime → Change runtime
type → T4 GPU`. Total time ~10–15 min (most of it is the one-time model download).

If any cell errors (model/library versions drift over time), copy the error to
Claude and it will patch this notebook.

## 0. Confirm the free GPU is on

In [ ]:
!nvidia-smi

## 1. Install SadTalker + download its models
_(one-time, a few minutes)_

In [ ]:
%cd /content
!git clone -q https://github.com/OpenTalker/SadTalker.git
%cd /content/SadTalker
# Install working library versions (SadTalker's pinned ones are too old for today's Colab).
!pip -q install "numpy==1.26.4" numba llvmlite resampy librosa soundfile imageio imageio-ffmpeg scikit-image kornia yacs pydub "face-alignment==1.3.5" safetensors av gfpgan basicsr facexlib edge-tts
!bash scripts/download_models.sh

## 1b. Two compatibility patches
Newer torchvision/numpy changed things SadTalker expects. These two lines fix it.

In [ ]:
import basicsr, os
# 1) torchvision moved functional_tensor -> functional
p = os.path.join(os.path.dirname(basicsr.__file__), 'data', 'degradations.py')
open(p, 'w').write(open(p).read().replace('functional_tensor', 'functional'))
# 2) restore numpy aliases (np.float etc.) for the inference subprocess
open("/content/SadTalker/sitecustomize.py", "w").write(
    "import numpy as np\n"
    "for a,t in [('float',float),('int',int),('bool',bool),('complex',complex),('object',object),('str',str)]:\n"
    "    hasattr(np,a) or setattr(np,a,t)\n")
print('patches applied')

## 2. Give Noria a voice line
Edit `NORIA_LINE` to whatever you want her to say. A warm neural voice is generated free.

In [ ]:
NORIA_LINE = "Hi, I'm Noria, your SkyGlobe companion. It's really good to finally meet you."
!edge-tts --voice en-US-JennyNeural --text "{NORIA_LINE}" --write-media /content/noria_audio.mp3
!ffmpeg -y -loglevel error -i /content/noria_audio.mp3 -ar 16000 -ac 1 /content/noria_audio.wav
print('voice ready')

## 3. Get Noria's face
This pulls her real render straight from your live site. For Noria-M, change the
filename to `noria-m.png`. (To use your own image instead, run `from google.colab
import files; files.upload()` and save it as `/content/noria.png`.)

In [ ]:
!wget -q -O /content/noria.png https://noria-body.onrender.com/assets/noria-f.png
from IPython.display import Image
Image('/content/noria.png', width=280)

## 4. Generate the talking video
_(~1–3 min. `--still` keeps a calm portrait; `--enhancer gfpgan` sharpens the face.)_

In [ ]:
%cd /content/SadTalker
!python inference.py \
  --driven_audio /content/noria_audio.wav \
  --source_image /content/noria.png \
  --result_dir /content/results \
  --still --preprocess full --enhancer gfpgan

## 5. Watch her — and download

In [ ]:
import glob, os
from IPython.display import HTML
from base64 import b64encode
vids = sorted(glob.glob('/content/results/**/*.mp4', recursive=True), key=os.path.getmtime)
assert vids, 'No video produced — check the previous cell output for an error.'
mp4 = vids[-1]
print('Talking Noria:', mp4)
data = b64encode(open(mp4, 'rb').read()).decode()
display(HTML(f'<video width=360 controls autoplay loop src="data:video/mp4;base64,{data}"></video>'))
try:
    from google.colab import files; files.download(mp4)
except Exception:
    print('Download it from the Files panel on the left:', mp4)